# English–Bangla Word Tokenization with PyTorch

This notebook builds an object-oriented word tokenizer, converts a parallel corpus into token IDs, creates padded PyTorch batches, and saves the results to Google Drive.

## Importing Libraries

In [ ]:
import json
import re
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


## Mounting Google Drive

In [ ]:
IN_COLAB = False

try:
    from google.colab import drive

    drive.mount("/content/drive")
    IN_COLAB = True
    print("Google Drive mounted.")
except ImportError:
    print("Not running in Google Colab; local storage will be used.")


Mounted at /content/drive
Google Drive mounted.


## Config

In [ ]:
@dataclass
class TranslationConfig:
    dataset_path: str
    output_dir: str
    batch_size: int = 32
    max_length: int = 32
    minimum_frequency: int = 1
    num_workers: int = 0
    source_column: str = "en"
    target_column: str = "bn"
    split_column: str = "split"


def find_dataset() -> Path:
    candidates = [
        Path("/content/en_bn_parallel_corpus_2k.tsv"),
        Path("en_bn_parallel_corpus_2k.tsv"),
        Path("/content/drive/MyDrive/Transformer/en_bn_parallel_corpus_2k.tsv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Upload en_bn_parallel_corpus_2k.tsv to Colab, the notebook folder, "
        "or MyDrive/Transformer/."
    )


dataset_path = find_dataset()

if IN_COLAB:
    output_dir = Path("/content/drive/MyDrive/Transformer/tokenized_data")
else:
    output_dir = Path("tokenized_data").resolve()

config = TranslationConfig(
    dataset_path=str(dataset_path),
    output_dir=str(output_dir),
)

print("Dataset:", config.dataset_path)
print("Output directory:", config.output_dir)

Dataset: /content/en_bn_parallel_corpus_2k.tsv
Output directory: /content/drive/MyDrive/Transformer/tokenized_data


## Special Tokens
- unk = the word does not exist in corpus/vocabulary
- pad = for matching sequence length
- bos = beginning of sentence
- eos = ending of setence

In [ ]:
class SpecialTokens:
    PAD = "<pad>"
    UNK = "<unk>"
    BOS = "<bos>"
    EOS = "<eos>"

    PAD_ID = 0
    UNK_ID = 1
    BOS_ID = 2
    EOS_ID = 3

    @classmethod
    def tokens(cls):
        return [cls.PAD, cls.UNK, cls.BOS, cls.EOS]

## Word tokenizer class

In [ ]:
class WordTokenizer:
    def __init__(self, language, lowercase=False, minimum_frequency=1):
        self.language = language
        self.lowercase = lowercase
        self.minimum_frequency = minimum_frequency
        self.id_to_token = SpecialTokens.tokens().copy()
        self.token_to_id = {
            token: index for index, token in enumerate(self.id_to_token)
        }
        self.token_counts = Counter()
        self.is_fitted = False

    @property
    def vocabulary_size(self):
        return len(self.id_to_token)

    def normalize(self, text):
        text = unicodedata.normalize("NFC", str(text).strip())
        return text.lower() if self.lowercase else text

    def tokenize(self, text):
        text = self.normalize(text)
        text = re.sub(r'([.,!?;:।()\[\]{}"“”])', r" \1 ", text)
        return text.split()

    def fit(self, texts):
        self.token_counts.clear()

        for text in texts:
            self.token_counts.update(self.tokenize(text))

        vocabulary = [
            token
            for token, frequency in self.token_counts.items()
            if frequency >= self.minimum_frequency
        ]
        vocabulary.sort(key=lambda token: (-self.token_counts[token], token))

        self.id_to_token = SpecialTokens.tokens() + vocabulary
        self.token_to_id = {
            token: index for index, token in enumerate(self.id_to_token)
        }
        self.is_fitted = True
        return self

    def encode(self, text, add_bos=True, add_eos=True, max_length=None):
        if not self.is_fitted:
            raise RuntimeError("Call fit() before encode().")

        token_ids = [
            self.token_to_id.get(token, SpecialTokens.UNK_ID)
            for token in self.tokenize(text)
        ]

        reserved = int(add_bos) + int(add_eos)
        if max_length is not None:
            token_ids = token_ids[: max(max_length - reserved, 0)]

        if add_bos:
            token_ids.insert(0, SpecialTokens.BOS_ID)
        if add_eos:
            token_ids.append(SpecialTokens.EOS_ID)

        return token_ids

    def decode(self, token_ids, skip_special_tokens=True):
        tokens = []

        for token_id in token_ids:
            if isinstance(token_id, torch.Tensor):
                token_id = token_id.item()

            token = (
                self.id_to_token[token_id]
                if 0 <= token_id < self.vocabulary_size
                else SpecialTokens.UNK
            )

            if skip_special_tokens and token in SpecialTokens.tokens():
                continue
            tokens.append(token)

        text = " ".join(tokens)
        text = re.sub(r"\s+([.,!?;:।)\]}])", r"\1", text)
        text = re.sub(r"([(\[{])\s+", r"\1", text)
        return text

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "language": self.language,
            "lowercase": self.lowercase,
            "minimum_frequency": self.minimum_frequency,
            "id_to_token": self.id_to_token,
            "token_counts": dict(self.token_counts),
        }
        path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    @classmethod
    def load(cls, path):
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        tokenizer = cls(
            language=payload["language"],
            lowercase=payload["lowercase"],
            minimum_frequency=payload["minimum_frequency"],
        )
        tokenizer.id_to_token = payload["id_to_token"]
        tokenizer.token_to_id = {
            token: index for index, token in enumerate(tokenizer.id_to_token)
        }
        tokenizer.token_counts = Counter(payload.get("token_counts", {}))
        tokenizer.is_fitted = True
        return tokenizer

## Parallel corpus class



In [ ]:
class ParallelCorpus:
    def __init__(self, config):
        self.config = config
        self.dataframe = None

    def load(self):
        self.dataframe = pd.read_csv(self.config.dataset_path, sep="\t")
        required = {
            self.config.split_column,
            self.config.source_column,
            self.config.target_column,
        }
        missing = required - set(self.dataframe.columns)

        if missing:
            raise ValueError(f"Dataset is missing columns: {sorted(missing)}")

        self.dataframe = self.dataframe.dropna(
            subset=[self.config.source_column, self.config.target_column]
        ).reset_index(drop=True)

        print(f"Loaded {len(self.dataframe)} sentence pairs.")
        return self

    def get_split(self, split_name):
        if self.dataframe is None:
            raise RuntimeError("Call load() before get_split().")

        available = set(self.dataframe[self.config.split_column].unique())
        if split_name not in available:
            raise ValueError(
                f"Unknown split '{split_name}'. Available splits: {sorted(available)}"
            )

        selected = self.dataframe[
            self.dataframe[self.config.split_column] == split_name
        ]
        return selected.copy().reset_index(drop=True)

    def show_split_sizes(self):
        print(self.dataframe[self.config.split_column].value_counts())

## Tokenized dataset class

Pre-encodes each English–Bangla pair and exposes it through PyTorch's Dataset interface.

In [ ]:
class TokenizedTranslationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        source_tokenizer,
        target_tokenizer,
        config,
        split_name,
    ):
        self.source_tokenizer = source_tokenizer
        self.target_tokenizer = target_tokenizer
        self.config = config
        self.split_name = split_name
        self.examples = []

        for index, row in dataframe.iterrows():
            source_text = str(row[config.source_column])
            target_text = str(row[config.target_column])
            self.examples.append(
                {
                    "id": int(index),
                    "split": split_name,
                    "source_text": source_text,
                    "target_text": target_text,
                    "source_ids": source_tokenizer.encode(
                        source_text, max_length=config.max_length
                    ),
                    "target_ids": target_tokenizer.encode(
                        target_text, max_length=config.max_length
                    ),
                }
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        return {
            "source": torch.tensor(example["source_ids"], dtype=torch.long),
            "target": torch.tensor(example["target_ids"], dtype=torch.long),
            "source_text": example["source_text"],
            "target_text": example["target_text"],
        }

    def save_jsonl(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        with path.open("w", encoding="utf-8") as file:
            for example in self.examples:
                file.write(json.dumps(example, ensure_ascii=False) + "\n")

        print(f"Saved {len(self.examples)} examples to {path}")


## Batch collator

Pads variable-length sequences to the longest sequence in each batch and builds masks that mark padding positions.

In [ ]:
class TranslationBatchCollator:
    def __init__(self, padding_id=SpecialTokens.PAD_ID):
        self.padding_id = padding_id

    def __call__(self, batch):
        sources = [example["source"] for example in batch]
        targets = [example["target"] for example in batch]

        padded_sources = pad_sequence(
            sources, batch_first=True, padding_value=self.padding_id
        )
        padded_targets = pad_sequence(
            targets, batch_first=True, padding_value=self.padding_id
        )

        return {
            "source": padded_sources,
            "target": padded_targets,
            "source_padding_mask": padded_sources.eq(self.padding_id),
            "target_padding_mask": padded_targets.eq(self.padding_id),
            "source_text": [example["source_text"] for example in batch],
            "target_text": [example["target_text"] for example in batch],
        }

## Data module

Coordinates corpus loading, training-only vocabulary fitting, split tokenization, DataLoader creation, and output saving.

In [ ]:
class TranslationDataModule:
    SPLITS = ("train", "validation", "test")

    def __init__(self, config):
        self.config = config
        self.corpus = ParallelCorpus(config)
        self.source_tokenizer = WordTokenizer(
            language="English",
            lowercase=True,
            minimum_frequency=config.minimum_frequency,
        )
        self.target_tokenizer = WordTokenizer(
            language="Bangla",
            lowercase=False,
            minimum_frequency=config.minimum_frequency,
        )
        self.datasets = {}
        self.collator = TranslationBatchCollator()

    def prepare(self):
        self.corpus.load().show_split_sizes()
        train_dataframe = self.corpus.get_split("train")

        # This fits vocabularies only; it does not train a neural network.
        self.source_tokenizer.fit(train_dataframe[self.config.source_column])
        self.target_tokenizer.fit(train_dataframe[self.config.target_column])

        print("English vocabulary:", self.source_tokenizer.vocabulary_size)
        print("Bangla vocabulary:", self.target_tokenizer.vocabulary_size)

        for split_name in self.SPLITS:
            self.datasets[split_name] = TokenizedTranslationDataset(
                dataframe=self.corpus.get_split(split_name),
                source_tokenizer=self.source_tokenizer,
                target_tokenizer=self.target_tokenizer,
                config=self.config,
                split_name=split_name,
            )
        return self

    def get_dataloader(self, split_name, shuffle=None):
        if split_name not in self.datasets:
            raise ValueError(f"Unknown or unprepared split: {split_name}")

        if shuffle is None:
            shuffle = split_name == "train"

        return DataLoader(
            self.datasets[split_name],
            batch_size=self.config.batch_size,
            shuffle=shuffle,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
        )

    def save_outputs(self):
        output_directory = Path(self.config.output_dir)
        output_directory.mkdir(parents=True, exist_ok=True)

        self.source_tokenizer.save(output_directory / "english_tokenizer.json")
        self.target_tokenizer.save(output_directory / "bangla_tokenizer.json")

        for split_name, dataset in self.datasets.items():
            dataset.save_jsonl(output_directory / f"{split_name}_tokenized.jsonl")

        metadata = {
            "config": asdict(self.config),
            "source_vocabulary_size": self.source_tokenizer.vocabulary_size,
            "target_vocabulary_size": self.target_tokenizer.vocabulary_size,
            "special_tokens": {
                "pad": {"token": SpecialTokens.PAD, "id": SpecialTokens.PAD_ID},
                "unk": {"token": SpecialTokens.UNK, "id": SpecialTokens.UNK_ID},
                "bos": {"token": SpecialTokens.BOS, "id": SpecialTokens.BOS_ID},
                "eos": {"token": SpecialTokens.EOS, "id": SpecialTokens.EOS_ID},
            },
            "examples": {
                split_name: len(dataset)
                for split_name, dataset in self.datasets.items()
            },
        }
        (output_directory / "tokenization_metadata.json").write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print("All outputs saved to:", output_directory)

## Prepare the tokenized datasets

Builds both vocabularies from the training split only and encodes the train, validation, and test sentences.

In [ ]:
data_module = TranslationDataModule(config).prepare()

Loaded 2000 sentence pairs.
split
train         1800
validation     100
test           100
Name: count, dtype: int64
English vocabulary: 449
Bangla vocabulary: 479


## Save outputs

Saves both tokenizer vocabularies, three tokenized JSONL splits, and reproducibility metadata to Drive or local storage.

In [ ]:
data_module.save_outputs()

Saved 1800 examples to /content/drive/MyDrive/Transformer/tokenized_data/train_tokenized.jsonl
Saved 100 examples to /content/drive/MyDrive/Transformer/tokenized_data/validation_tokenized.jsonl
Saved 100 examples to /content/drive/MyDrive/Transformer/tokenized_data/test_tokenized.jsonl
All outputs saved to: /content/drive/MyDrive/Transformer/tokenized_data


## Create and inspect DataLoaders

Creates shuffled training batches and deterministic validation/test batches, then displays tensor and mask shapes.

In [ ]:
train_loader = data_module.get_dataloader("train")
validation_loader = data_module.get_dataloader("validation")
test_loader = data_module.get_dataloader("test")

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

batch = next(iter(train_loader))
print("Source shape:", tuple(batch["source"].shape))
print("Target shape:", tuple(batch["target"].shape))
print("Source mask shape:", tuple(batch["source_padding_mask"].shape))
print("Target mask shape:", tuple(batch["target_padding_mask"].shape))


Train batches: 57
Validation batches: 4
Test batches: 4
Source shape: (32, 13)
Target shape: (32, 11)
Source mask shape: (32, 13)
Target mask shape: (32, 11)


## Verify encoding and decoding

Decodes one batch example so you can confirm that token IDs still represent the original sentences.

In [ ]:
source_ids = batch["source"][0]
target_ids = batch["target"][0]

print("Original English:", batch["source_text"][0])
print("Decoded English:", data_module.source_tokenizer.decode(source_ids))
print("Original Bangla:", batch["target_text"][0])
print("Decoded Bangla:", data_module.target_tokenizer.decode(target_ids))

Original English: The student studies chemistry in the afternoon.
Decoded English: the student studies chemistry in the afternoon.
Original Bangla: শিক্ষার্থী রসায়ন বিকেলে পড়ে।
Decoded Bangla: শিক্ষার্থী রসায়ন বিকেলে পড়ে।


## Prepare decoder inputs and labels

Shifts the target sequence by one position: the decoder reads all but the last token and predicts all but the first token.

In [ ]:
decoder_input = batch["target"][:, :-1]
target_labels = batch["target"][:, 1:]

print("Full target shape:", tuple(batch["target"].shape))
print("Decoder input shape:", tuple(decoder_input.shape))
print("Target label shape:", tuple(target_labels.shape))


Full target shape: (32, 11)
Decoder input shape: (32, 10)
Target label shape: (32, 10)
